In [ ]:
import pandas as pd
import os
from datetime import datetime, timedelta

# ===============================
# PARAMETER
# ===============================

start_date = "2024-01-01"
end_date   = "2026-02-19"

folder_path = "data_harian"
output_file = "bapanas harga konsumen 2024-skrng.xlsx"

# ===============================
# KOLOM KOMODITAS
# ===============================

kolom_komoditas = [
    "Beras Premium",
    "Beras Medium",
    "Beras SPHP",
    "Beras Medium Non SPHP",
    "Beras Khusus (Lokal)",
    "Jagung Tk Peternak",
    "Kedelai Biji Kering (Impor)",
    "Bawang Merah",
    "Bawang Putih Bonggol",
    "Cabai Merah Keriting",
    "Cabai Merah Besar",
    "Daging Sapi Murni",
    "Cabai Rawit Merah",
    "Daging Ayam Ras",
    "Telur Ayam Ras",
    "Gula Konsumsi",
    "Minyak Goreng Kemasan",
    "Minyak Goreng Curah",
    "Tepung Terigu (Curah)",
    "Minyakita",
    "Tepung Terigu Kemasan",
    "Ikan Kembung",
    "Ikan Tongkol",
    "Ikan Bandeng",
    "Garam Konsumsi",
    "Daging Kerbau Beku (Impor Luar Negeri)",
    "Daging Kerbau Segar (Lokal)"
]

# ===============================
# FUNGSI GENERATE TANGGAL
# ===============================

def generate_dates(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    delta = end - start

    return [
        (start + timedelta(days=i)).strftime("%Y-%m-%d")
        for i in range(delta.days + 1)
    ]

# ===============================
# PROSES
# ===============================
rerata_manual_list = []
all_data = {}
jatim_data = []

tanggal_list = generate_dates(start_date, end_date)

for tanggal in tanggal_list:

    file_name = f"{tanggal}.xlsx"
    file_path = os.path.join(folder_path, file_name)

    if not os.path.exists(file_path):
        print(f"File tidak ditemukan: {file_name}")
        continue

    print(f"Memproses: {file_name}")

    df = pd.read_excel(file_path)

    # ===============================
    # BAGIAN 1: DATA PER KOTA (KONSUMEN)
    # ===============================

    df_konsumen = df[df["tipe"] == "konsumen"].copy()

    df_konsumen = df_konsumen[df_konsumen["Kota/Kabupaten"].notna()]
    df_konsumen = df_konsumen[
        ~df_konsumen["Kota/Kabupaten"]
        .astype(str)
        .str.contains("Grand|Rerata|HET|Tertinggi|Terendah", case=False, na=False)
    ]

    if df_konsumen.empty:
        print(f"Tidak ada data konsumen pada {tanggal} → skip tanggal")
        continue

    df_konsumen = df_konsumen[["Kota/Kabupaten"] + kolom_komoditas]

    # Bersihkan angka
    for col in kolom_komoditas:
        df_konsumen[col] = (
            df_konsumen[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .replace("-", None)
        )
        df_konsumen[col] = pd.to_numeric(df_konsumen[col], errors="coerce")

    df_konsumen["Tanggal"] = tanggal
    df_konsumen = df_konsumen[["Kota/Kabupaten", "Tanggal"] + kolom_komoditas]

    # Simpan per kota
    for _, row in df_konsumen.iterrows():
        kota = row["Kota/Kabupaten"]
        row_df = pd.DataFrame([row.drop("Kota/Kabupaten")])

        if kota not in all_data:
            all_data[kota] = row_df
        else:
            all_data[kota] = pd.concat(
                [all_data[kota], row_df],
                ignore_index=True
            )

    # ===============================
    # BAGIAN 2: DATA JATIM (AMBIL GRAND TOTAL)
    # ===============================

    df_jatim = df[
        df["Kota/Kabupaten"]
        .astype(str)
        .str.contains("Grand Total", case=False, na=False)
    ]

    if df_jatim.empty:
        print(f"Tidak ada Grand Total pada {tanggal} → skip jatim")
        continue

    df_jatim = df_jatim[["Kota/Kabupaten"] + kolom_komoditas]

    # Bersihkan angka
    for col in kolom_komoditas:
        df_jatim[col] = (
            df_jatim[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .replace("-", None)
        )
        df_jatim[col] = pd.to_numeric(df_jatim[col], errors="coerce")

    df_jatim["Tanggal"] = tanggal
    df_jatim = df_jatim[["Tanggal"] + kolom_komoditas]

    jatim_data.append(df_jatim)

    # ===============================
    # BAGIAN 3: RERATA JATIM MANUAL
    # ===============================

    df_rerata = df_konsumen.copy()

    # Buang nilai 0 (dianggap tidak valid)
    for col in kolom_komoditas:
        df_rerata[col] = df_rerata[col].replace(0, pd.NA)

    # Hitung rata-rata per kolom (skipna otomatis)
    # rerata_row = df_rerata[kolom_komoditas].mean(skipna=True)
    rerata_row = df_rerata[kolom_komoditas].mean()


    rerata_row["Tanggal"] = tanggal

    rerata_manual_list.append(pd.DataFrame([rerata_row]))

# ===============================
# SIMPAN KE EXCEL
# ===============================

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

    # Sheet per kota
    for kota, data_kota in all_data.items():
        data_kota = data_kota.sort_values("Tanggal").reset_index(drop=True)
        data_kota.insert(0, "No", range(1, len(data_kota) + 1))
        data_kota.to_excel(writer, sheet_name=kota[:31], index=False)

    # Sheet JATIM
    if jatim_data:
        df_jatim_final = pd.concat(jatim_data, ignore_index=True)

        # 🔥 HAPUS BARIS JIKA BERAS MEDIUM DAN PREMIUM SAMA-SAMA NaN
        df_jatim_final = df_jatim_final[~(
            df_jatim_final["Beras Medium"].isna() &
                df_jatim_final["Beras Premium"].isna()
            )]
        df_jatim_final = df_jatim_final.sort_values("Tanggal").reset_index(drop=True)
        df_jatim_final.insert(0, "No", range(1, len(df_jatim_final) + 1))

        df_jatim_final.to_excel(writer, sheet_name="jatim", index=False)
        print("Selesai! File disimpan:", output_file)
    
    # ===============================
    # Sheet RERATA JATIM MANUAL
    # ===============================

    if rerata_manual_list:
        df_rerata_final = pd.concat(rerata_manual_list, ignore_index=True)

        df_rerata_final = df_rerata_final.sort_values("Tanggal").reset_index(drop=True)
        df_rerata_final.insert(0, "No", range(1, len(df_rerata_final) + 1))

        df_rerata_final = df_rerata_final[["No", "Tanggal"] + kolom_komoditas]

        df_rerata_final.to_excel(
            writer,
            sheet_name="rerata jatim manual",
            index=False
        )

Memproses: 2024-01-01.xlsx
Memproses: 2024-01-02.xlsx
Memproses: 2024-01-03.xlsx
Memproses: 2024-01-04.xlsx
Memproses: 2024-01-05.xlsx
Memproses: 2024-01-06.xlsx
Memproses: 2024-01-07.xlsx
Memproses: 2024-01-08.xlsx
Memproses: 2024-01-09.xlsx
Memproses: 2024-01-10.xlsx
Memproses: 2024-01-11.xlsx
Memproses: 2024-01-12.xlsx
Memproses: 2024-01-13.xlsx
Memproses: 2024-01-14.xlsx
Memproses: 2024-01-15.xlsx
Memproses: 2024-01-16.xlsx
Memproses: 2024-01-17.xlsx
Memproses: 2024-01-18.xlsx
Memproses: 2024-01-19.xlsx
Memproses: 2024-01-20.xlsx
Memproses: 2024-01-21.xlsx
Memproses: 2024-01-22.xlsx
Memproses: 2024-01-23.xlsx
Memproses: 2024-01-24.xlsx
Memproses: 2024-01-25.xlsx
Memproses: 2024-01-26.xlsx
Memproses: 2024-01-27.xlsx
Memproses: 2024-01-28.xlsx
Memproses: 2024-01-29.xlsx
Memproses: 2024-01-30.xlsx
Memproses: 2024-01-31.xlsx
Memproses: 2024-02-01.xlsx
Memproses: 2024-02-02.xlsx
Memproses: 2024-02-03.xlsx
Memproses: 2024-02-04.xlsx
Memproses: 2024-02-05.xlsx
Memproses: 2024-02-06.xlsx
M

In [ ]:
import pandas as pd
import os
from datetime import datetime, timedelta

# ===============================
# PARAMETER
# ===============================

start_date = "2024-01-01"
end_date   = "2026-02-19"

folder_path = "data_harian"
output_file = "bapanas harga produsen 2024-skrng.xlsx"

# ===============================
# KOLOM KOMODITAS
# ===============================

kolom_komoditas = [
    "GKP Tingkat Petani",
    "GKG Tingkat Penggilingan",
    "Beras Medium Penggilingan",
    "Beras Premium Penggilingan",
    "Jagung Pipilan Kering",
    "Kedelai Biji Kering (Lokal)",
    "Sapi (Hidup)",
    "Ayam Ras Pedaging (Hidup)",
    "Gula Konsumsi di Petani/Pabrik Gula"
]

# ===============================
# FUNGSI GENERATE TANGGAL
# ===============================

def generate_dates(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    delta = end - start

    return [
        (start + timedelta(days=i)).strftime("%Y-%m-%d")
        for i in range(delta.days + 1)
    ]

# ===============================
# PROSES
# ===============================
rerata_manual_list = []
all_data = {}
jatim_data = []

tanggal_list = generate_dates(start_date, end_date)

for tanggal in tanggal_list:

    file_name = f"{tanggal}.xlsx"
    file_path = os.path.join(folder_path, file_name)

    if not os.path.exists(file_path):
        print(f"File tidak ditemukan: {file_name}")
        continue

    print(f"Memproses: {file_name}")

    df = pd.read_excel(file_path)

    # ===============================
    # BAGIAN 1: DATA PER KOTA (produsen)
    # ===============================

    df_produsen = df[df["tipe"] == "produsen"].copy()

    df_produsen = df_produsen[df_produsen["Kota/Kabupaten"].notna()]
    df_produsen = df_produsen[
        ~df_produsen["Kota/Kabupaten"]
        .astype(str)
        .str.contains("Grand|Rerata|HET|Tertinggi|Terendah", case=False, na=False)
    ]

    if df_produsen.empty:
        print(f"Tidak ada data produsen pada {tanggal} → skip tanggal")
        continue

    df_produsen = df_produsen[["Kota/Kabupaten"] + kolom_komoditas]

    # Bersihkan angka
    for col in kolom_komoditas:
        df_produsen[col] = (
            df_produsen[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .replace("-", None)
        )
        df_produsen[col] = pd.to_numeric(df_produsen[col], errors="coerce")

    df_produsen["Tanggal"] = tanggal
    df_produsen = df_produsen[["Kota/Kabupaten", "Tanggal"] + kolom_komoditas]

    # Simpan per kota
    for _, row in df_produsen.iterrows():
        kota = row["Kota/Kabupaten"]
        row_df = pd.DataFrame([row.drop("Kota/Kabupaten")])

        if kota not in all_data:
            all_data[kota] = row_df
        else:
            all_data[kota] = pd.concat(
                [all_data[kota], row_df],
                ignore_index=True
            )

    # ===============================
    # BAGIAN 2: DATA JATIM (AMBIL GRAND TOTAL)
    # ===============================

    df_jatim = df[
        df["Kota/Kabupaten"]
        .astype(str)
        .str.contains("Grand Total", case=False, na=False)
    ]

    if df_jatim.empty:
        print(f"Tidak ada Grand Total pada {tanggal} → skip jatim")
        continue

    df_jatim = df_jatim[["Kota/Kabupaten"] + kolom_komoditas]

    # Bersihkan angka
    for col in kolom_komoditas:
        df_jatim[col] = (
            df_jatim[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .replace("-", None)
        )
        df_jatim[col] = pd.to_numeric(df_jatim[col], errors="coerce")

    df_jatim["Tanggal"] = tanggal
    df_jatim = df_jatim[["Tanggal"] + kolom_komoditas]

    jatim_data.append(df_jatim)

    # ===============================
    # BAGIAN 3: RERATA JATIM MANUAL
    # ===============================

    df_rerata = df_produsen.copy()

    # Buang nilai 0 (dianggap tidak valid)
    for col in kolom_komoditas:
        df_rerata[col] = df_rerata[col].replace(0, pd.NA)

    # Hitung rata-rata per kolom (skipna otomatis)
    # rerata_row = df_rerata[kolom_komoditas].mean(skipna=True)
    rerata_row = df_rerata[kolom_komoditas].mean()


    rerata_row["Tanggal"] = tanggal

    rerata_manual_list.append(pd.DataFrame([rerata_row]))

# ===============================
# SIMPAN KE EXCEL
# ===============================

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

    # Sheet per kota
    for kota, data_kota in all_data.items():
        data_kota = data_kota.sort_values("Tanggal").reset_index(drop=True)
        data_kota.insert(0, "No", range(1, len(data_kota) + 1))
        data_kota.to_excel(writer, sheet_name=kota[:31], index=False)

    # Sheet JATIM
    if jatim_data:
        df_jatim_final = pd.concat(jatim_data, ignore_index=True)

        # # 🔥 HAPUS BARIS JIKA BERAS MEDIUM DAN PREMIUM SAMA-SAMA NaN
        # df_jatim_final = df_jatim_final[~(
        #     df_jatim_final["Beras Medium"].isna() &
        #         df_jatim_final["Beras Premium"].isna()
        #     )]
        df_jatim_final = df_jatim_final.sort_values("Tanggal").reset_index(drop=True)
        df_jatim_final.insert(0, "No", range(1, len(df_jatim_final) + 1))

        df_jatim_final.to_excel(writer, sheet_name="jatim", index=False)
        print("Selesai! File disimpan:", output_file)
    
    # ===============================
    # Sheet RERATA JATIM MANUAL
    # ===============================

    if rerata_manual_list:
        df_rerata_final = pd.concat(rerata_manual_list, ignore_index=True)

        df_rerata_final = df_rerata_final.sort_values("Tanggal").reset_index(drop=True)
        df_rerata_final.insert(0, "No", range(1, len(df_rerata_final) + 1))

        df_rerata_final = df_rerata_final[["No", "Tanggal"] + kolom_komoditas]

        df_rerata_final.to_excel(
            writer,
            sheet_name="rerata jatim manual",
            index=False
        )

Rata-rata Beras Premium : 14480.69
Rata-rata Beras Medium  : 12788.97


/tmp/ipykernel_23130/3690856085.py:44: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("-", np.nan, inplace=True)
